# 1일차 파트 1 실습: LLM의 원리와 한계

실습은 한 개입니다(실습 1). 질문 카드 다섯 장을 두 모델에 보내고, 응답을 읽어 사용 기준표를 만듭니다.
`NotImplementedError`가 있는 곳과 빈 문자열(`''`)을 직접 채웁니다. 막히면 힌트를 펼쳐 보고, 다 한 뒤에 `1일차_실습_풀이.ipynb`와 비교합니다.
테스트 셀은 기준표의 형식과 계산 정답만 확인합니다. 판정과 이유는 응답을 읽은 여러분이 정합니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/1일차'


def read_material(name):
    """재료 폴더(data/1일차)의 파일을 읽어 문자열로 돌려줍니다."""
    return open(f'{DATA}/{name}', encoding='utf-8').read().strip()


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 1 질문 카드로 사용 기준표 만들기

**목적** 질문 유형마다 모델의 응답을 어떻게 쓸지 직접 정해 봅니다. 응답이 맞았는지와 응답을 어떻게 쓸지는 별개라는 점을 확인합니다.

**재료** `data/1일차/질문카드.txt`. 정의, 원리, 계산, 최신 정보, 사내 규격 순서의 질문 다섯 개입니다. 교재의 예제 1-6과 유형은 같고 주제가 다릅니다.

**할 일**

1. 계산 카드의 정답을 코드로 구합니다. 모델에 보내기 전에 정답을 먼저 알아야 응답을 판정할 수 있습니다.
2. 카드 다섯 장을 소형 모델과 기본 모델에 보냅니다(준비된 셀을 실행합니다). 사내 규격 카드에는 실제 수치나 설비 이름이 없어서 보낼 수 있습니다. 모델이 모르는 내용에 무엇이라고 답하는지 보려고 보냅니다.
3. 응답을 읽고 사용 기준표 `table`을 채웁니다. 모델별 판정, 사용 기준, 이유 한 줄입니다.
4. 최신 정보 카드에 "근거를 확인할 수 없는 내용은 '확인 불가'라고 적어줘" 한 줄을 붙여 다시 보내고, 응답이 어떻게 달라졌는지 적습니다.
5. 사내 규격 카드의 응답에서 그대로 쓰면 위험한 문장을 하나 골라 적습니다.

**완료 기준** 테스트 셀이 "실습 1 통과"를 출력합니다. 4번과 5번은 "적어 보기" 셀에 적습니다.

<details><summary>힌트 1</summary>

누적 수율은 단계별 수율을 모두 곱한 값입니다. 퍼센트를 100으로 나누어 곱한 뒤 다시 100을 곱하고, `round(값, 2)`로 소수 둘째 자리까지 구합니다.
</details>

<details><summary>힌트 2</summary>

판정은 응답이 맞았는지입니다. 최신 정보처럼 맞는지 확인할 근거가 응답 밖에 있으면 "판정 불가"입니다. 모델이 모른다고 답하고 일반론을 덧붙였을 때도 "판정 불가"로 적습니다. 덧붙인 일반론이 우리 라인에 맞는지 알 수 없기 때문입니다. 사용 기준은 응답이 맞았는지가 아니라, 맞는지 무엇으로 확인할 수 있는지로 정합니다. 정의와 원리 카드는 여러분이 읽고 판정할 수 있으면 "읽고 판정해 쓴다", 판정할 수 없는 분야라면 "검증하고 쓴다"입니다. 교재의 "질문 유형별 사용 기준" 절의 표를 참고합니다.
</details>

In [ ]:
# 실습 1 재료 (그대로 실행합니다)
import checks_d1

cards = {}
for block in read_material('질문카드.txt').split('\n\n'):
    title, _, body = block.partition('\n')
    cards[title.strip('[]')] = body
for kind, card in cards.items():
    print(f'[{kind}] {card}\n')

In [ ]:
# 실습 1 (1) 계산 카드의 정답을 코드로 구합니다
def cumulative_yield(step_yields):
    """단계별 수율(%)의 목록을 받아 누적 수율(%)을 소수 둘째 자리까지 돌려줍니다."""
    raise NotImplementedError('이 부분을 채웁니다')


correct_yield = cumulative_yield([98.5, 97.2, 99.1, 96.8])
print(f'코드로 구한 정답: {correct_yield}%')

In [ ]:
# 실습 1 (2) 카드 다섯 장을 두 모델에 보냅니다 (실제 API를 부릅니다. 10회)
for kind, card in cards.items():
    for model in ['소형', '기본']:
        print(f'===== {kind}, {model} 모델 =====')
        llm.Chat(model=model).say(card)
        print()

In [ ]:
# 실습 1 (3) 응답을 읽고 사용 기준표를 채웁니다
# 판정: '맞음', '틀림', '판정 불가'
# 기준: '읽고 판정해 쓴다', '검증하고 쓴다', '도구에 맡긴다', '입력하지 않는다'
table = {
    '정의':      {'소형': '', '기본': '', '기준': '', '이유': ''},
    '원리':      {'소형': '', '기본': '', '기준': '', '이유': ''},
    '계산':      {'소형': '', '기본': '', '기준': '', '이유': ''},
    '최신 정보': {'소형': '', '기본': '', '기준': '', '이유': ''},
    '사내 규격': {'소형': '', '기본': '', '기준': '', '이유': ''},
}

In [ ]:
# 실습 1 확인 (API를 부르지 않습니다)
test_results = checks_d1.check_table(table, correct_yield)

In [ ]:
# 실습 1 (4) 최신 정보 카드에 한 줄을 붙여 다시 보냅니다 (실제 API를 부릅니다. 2회)
rule = "\n근거를 확인할 수 없는 내용은 '확인 불가'라고 적어줘."
for model in ['소형', '기본']:
    print(f'===== 최신 정보 + 한 줄, {model} 모델 =====')
    llm.Chat(model=model).say(cards['최신 정보'] + rule)
    print()

**적어 보기**

- 한 줄을 붙이기 전과 후에 최신 정보 카드의 응답이 어떻게 달라졌나요? (모델별로)
- 이 응답을 보고서에 옮기기 전에 확인할 것:
- 사내 규격 카드의 응답에서 그대로 쓰면 위험한 문장(모델 이름과 함께):
- 그 문장이 위험한 이유:

## 더 해 보기

- 내 업무의 질문으로 카드 다섯 장을 만들어 같은 순서(정의, 원리, 계산, 최신 정보, 사내 규격)로 기준표를 만들어 봅니다. 실제 회사 이름, 설비 번호, 사내 수치는 어느 카드에도 넣지 않습니다.
- 계산 카드를 세 번 보내 첫 줄의 숫자가 세 번 모두 같은지 봅니다.
- 원리 카드의 응답에서 사실인지 확신이 서지 않는 문장을 하나 골라, 무엇으로 확인할 수 있을지 적어 봅니다.